# DENTEX 4-Class Abnormal Tooth Detection — Exploratory Data Analysis (EDA)

This notebook provides a publication-grade Exploratory Data Analysis of the **DENTEX Challenge (Abnormal Tooth Detection)** dataset across 4 clinical pathology classes:
1. **Caries** (`#E69F00` - Orange)
2. **Deep Caries** (`#D55E00` - Vermilion)
3. **Periapical Lesion** (`#0072B2` - Blue)
4. **Impacted Tooth** (`#009E73` - Green)

> **Note on Maintenance & Generation**: This notebook is self-contained and can be executed directly in Jupyter, VS Code, or Google Colab. It can also be programmatically regenerated or synchronized via `python scripts/build_eda_notebook.py`.

---
### Notebook Outline:
1. **Environment Setup & Data Resolution**: Auto-detection of Google Colab / Google Drive vs local directory.
2. **Annotation Ingestion & Dataset Integrity**: Loading COCO annotations, validating file paths, dimensions, and bounding boxes.
3. **Class Distribution & Imbalance Analysis**: Bar and pie charts, imbalance ratio calculation, multi-morbidity (lesions per scan).
4. **Bounding Box Geometry & COCO Size Taxonomy**: Small (< 32²), Medium (32²–96²), Large (>= 96²), and aspect ratio distributions.
5. **2D Spatial Pathology Heatmap on Panoramic Dental Arch**: Anatomical spatial heatmaps on normalized dental panoramic coordinates $(x_c, y_c) \in [0, 1] \times [0, 1]$.
6. **Pathology Co-occurrence & Correlation Matrix**: Cross-pathology co-occurrence frequency, conditional probabilities, and disease progression patterns.
7. **Radiographic Properties & Pixel Intensity Distribution**: Grayscale histograms, contrast metrics, dynamic range across dental X-rays.
8. **Clinical Sample Visualizer**: Ground truth visualization with color-coded bounding boxes and clinical labels.


In [ ]:
import os
import sys
import json
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import cv2

# Set publication style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8

# Environment auto-detection: Google Colab vs Local
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/tmanhococ/dentex-benchmark.git'
BRANCH = 'main'

if IN_COLAB:
    import subprocess
    # 1. Mount Google Drive
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f'Drive mount info/warning: {e}')

    # 2. Locate or clone dentex-benchmark repository
    candidates = [
        Path('/content/dentex-benchmark'),
        Path('/content/drive/MyDrive/dentex-benchmark'),
        Path.cwd(),
        Path.cwd().parent
    ]
    repo_root = None
    for cand in candidates:
        if (cand / 'dentex_bench').exists() or (cand / 'data').exists():
            repo_root = cand.resolve()
            break

    if repo_root is None:
        target_dir = Path('/content/dentex-benchmark')
        print(f'🚀 Cloning {REPO_URL} into {target_dir}...')
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(target_dir)], check=True)
        repo_root = target_dir

    os.chdir(repo_root)
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

    drive_base = Path('/content/drive/MyDrive/dentex-benchmark')
    if (drive_base / 'data').exists():
        DATA_DIR = drive_base / 'data'
    elif (repo_root / 'data').exists():
        DATA_DIR = repo_root / 'data'
    else:
        DATA_DIR = Path('data')
else:
    # Ensure project root is in sys.path for local workspace
    project_root = Path.cwd().resolve()
    if project_root.name == 'notebooks':
        project_root = project_root.parent
        os.chdir(project_root)
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    DATA_DIR = Path('data')

TRAIN_JSON = DATA_DIR / 'processed' / 'instances_train.json'
VAL_JSON = DATA_DIR / 'processed' / 'instances_val.json'
RAW_IMG_DIR = DATA_DIR / 'raw'

# Clinical 4-class ontology and publication color scheme
CLASS_METADATA = {
    1: {'name': 'Caries', 'color': '#E69F00', 'description': 'Superficial or moderate enamel/dentin decay'},
    2: {'name': 'Deep Caries', 'color': '#D55E00', 'description': 'Severe caries approximating or penetrating dental pulp'},
    3: {'name': 'Periapical Lesion', 'color': '#0072B2', 'description': 'Inflammatory radiolucency at the tooth root apex'},
    4: {'name': 'Impacted Tooth', 'color': '#009E73', 'description': 'Unerupted or malpositioned tooth prevented from erupting'}
}
CLASS_NAMES = [CLASS_METADATA[i]['name'] for i in sorted(CLASS_METADATA.keys())]
CLASS_COLORS = [CLASS_METADATA[i]['color'] for i in sorted(CLASS_METADATA.keys())]
COLOR_MAP = {CLASS_METADATA[i]['name']: CLASS_METADATA[i]['color'] for i in CLASS_METADATA}

print(f"Running in: {'Google Colab' if IN_COLAB else 'Local Workstation'}")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Train annotations: {TRAIN_JSON} (Exists: {TRAIN_JSON.exists()})")
print(f"Val annotations:   {VAL_JSON} (Exists: {VAL_JSON.exists()})")
print(f"Raw images dir:    {RAW_IMG_DIR} (Exists: {RAW_IMG_DIR.exists()})")


## 1. Annotation Ingestion & Dataset Integrity Report

We ingest both `instances_train.json` and `instances_val.json` COCO annotations, cross-reference image files on disk, and validate bounding box geometry (non-negative coordinates, non-zero dimensions, and within image boundaries).


In [ ]:
def load_and_validate_coco(json_path: Path, split_name: str, raw_dir: Path) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Loads COCO json and builds a validated tabular representation of annotations."""
    if not json_path.exists():
        raise FileNotFoundError(f"Annotation file not found: {json_path}")
        
    with open(json_path, 'r', encoding='utf-8') as f:
        coco = json.load(f)
        
    images_dict = {im['id']: im for im in coco.get('images', [])}
    categories_dict = {cat['id']: cat['name'] for cat in coco.get('categories', [])}
    
    rows = []
    missing_files = 0
    invalid_bboxes = 0
    
    for ann in coco.get('annotations', []):
        img_id = ann['image_id']
        im_meta = images_dict.get(img_id, {})
        fname = im_meta.get('file_name', '')
        w = im_meta.get('width', 0)
        h = im_meta.get('height', 0)
        
        # Check image existence on disk
        if raw_dir.exists() and not (raw_dir / fname).exists():
            missing_files += 1
            
        bx, by, bw, bh = ann['bbox']
        # Validate bounding box coordinates
        if bw <= 0 or bh <= 0 or bx < 0 or by < 0:
            invalid_bboxes += 1
            
        area = ann.get('area', bw * bh)
        cid = ann['category_id']
        cname = categories_dict.get(cid, CLASS_METADATA.get(cid, {}).get('name', f'Class {cid}'))
        
        xc = bx + bw / 2.0
        yc = by + bh / 2.0
        
        rows.append({
            'ann_id': ann['id'],
            'image_id': img_id,
            'file_name': fname,
            'image_width': w,
            'image_height': h,
            'category_id': cid,
            'category_name': cname,
            'bbox_x': bx,
            'bbox_y': by,
            'bbox_w': bw,
            'bbox_h': bh,
            'bbox_area': area,
            'aspect_ratio': bw / max(bh, 1e-4),
            'x_center_norm': xc / max(w, 1) if w > 0 else 0.5,
            'y_center_norm': yc / max(h, 1) if h > 0 else 0.5,
            'split': split_name
        })
        
    df = pd.DataFrame(rows)
    stats = {
        'split': split_name,
        'num_images': len(images_dict),
        'num_annotations': len(rows),
        'missing_image_files': missing_files,
        'invalid_bounding_boxes': invalid_bboxes
    }
    return df, stats

df_train, stats_train = load_and_validate_coco(TRAIN_JSON, 'train', RAW_IMG_DIR)
df_val, stats_val = load_and_validate_coco(VAL_JSON, 'val', RAW_IMG_DIR)
df_all = pd.concat([df_train, df_val], ignore_index=True)

# Integrity Summary Table
summary_df = pd.DataFrame([stats_train, stats_val])
print('=== DENTEX Dataset Integrity Summary ===')
display(summary_df) if 'display' in globals() else print(summary_df.to_string(index=False))
print(f'\nTotal Annotations Loaded: {len(df_all):,}')
print(f'Total Unique Scans: {df_all["image_id"].nunique():,}')


## 2. Class Distribution, Multi-Morbidity & Class Imbalance

We analyze the prevalence across the 4 abnormal tooth categories: **Caries**, **Deep Caries**, **Periapical Lesion**, and **Impacted Tooth**.
We calculate the class imbalance ratio ($\rho = \frac{\max(N_c)}{\min(N_c)}$) and inspect the distribution of pathology lesions per scan.


In [ ]:
# Class counts by split
class_split_counts = pd.crosstab(df_all['category_name'], df_all['split'])
for c in CLASS_NAMES:
    if c not in class_split_counts.index:
        class_split_counts.loc[c] = 0
class_split_counts = class_split_counts.reindex(CLASS_NAMES).fillna(0).astype(int)
class_split_counts['Total'] = class_split_counts.sum(axis=1)

# Compute Imbalance Ratio
max_c = class_split_counts['Total'].max()
min_c = max(class_split_counts['Total'].min(), 1)
imbalance_ratio = max_c / min_c

print('=== Pathology Class Breakdown ===')
print(class_split_counts)
print(f'\nImbalance Ratio (Max / Min): {imbalance_ratio:.2f}:1')

# Visualizations: 3 subplots (Split counts, Total Pie, Lesions per scan)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Subplot 1: Grouped Bar Chart of Train vs Val
x_pos = np.arange(len(CLASS_NAMES))
bar_w = 0.35
train_counts = [int(class_split_counts.loc[c, 'train']) if 'train' in class_split_counts else 0 for c in CLASS_NAMES]
val_counts = [int(class_split_counts.loc[c, 'val']) if 'val' in class_split_counts else 0 for c in CLASS_NAMES]

axes[0].bar(x_pos - bar_w/2, train_counts, width=bar_w, label='Train', color='#2b5c8f', alpha=0.9, edgecolor='black')
axes[0].bar(x_pos + bar_w/2, val_counts, width=bar_w, label='Val', color='#e27c38', alpha=0.9, edgecolor='black')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(CLASS_NAMES, rotation=20, ha='right', fontweight='medium')
axes[0].set_ylabel('Number of Annotations')
axes[0].set_title('Pathology Frequency by Dataset Split', fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# Subplot 2: Donut Chart of Overall Distribution
total_counts = [int(class_split_counts.loc[c, 'Total']) for c in CLASS_NAMES]
axes[1].pie(
    total_counts,
    labels=CLASS_NAMES,
    colors=[COLOR_MAP[c] for c in CLASS_NAMES],
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops=dict(width=0.6, edgecolor='white', linewidth=2)
)
axes[1].set_title(f'Overall Pathology Distribution\n(Imbalance Ratio: {imbalance_ratio:.2f}:1)', fontweight='bold')

# Subplot 3: Multi-Morbidity (Lesions per scan)
lesions_per_scan = df_all.groupby(['split', 'image_id']).size().reset_index(name='lesion_count')
max_lesions = max(int(lesions_per_scan['lesion_count'].max()), 5) if len(lesions_per_scan) > 0 else 5
bins = np.arange(0.5, max_lesions + 1.5, 1)

train_lesions = lesions_per_scan[lesions_per_scan['split'] == 'train']['lesion_count'].tolist()
val_lesions = lesions_per_scan[lesions_per_scan['split'] == 'val']['lesion_count'].tolist()

axes[2].hist(
    [train_lesions, val_lesions],
    bins=bins,
    label=['Train', 'Val'],
    color=['#2b5c8f', '#e27c38'],
    alpha=0.85,
    edgecolor='black'
)
axes[2].set_xticks(range(1, int(max_lesions) + 1))
axes[2].set_xlabel('Pathologies per Panoramic Radiograph')
axes[2].set_ylabel('Number of Scans')
axes[2].set_title('Multi-Morbidity Distribution per Patient Scan', fontweight='bold')
axes[2].legend()
axes[2].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


## 3. Bounding Box Geometry & COCO Size Taxonomy

Detection frameworks evaluate models according to the standard COCO scale thresholds:
- **Small ($S$)**: $\text{area} < 32^2 = 1024 \text{ px}^2$
- **Medium ($M$)**: $32^2 \le \text{area} < 96^2 = 9216 \text{ px}^2$
- **Large ($L$)**: $\text{area} \ge 96^2 = 9216 \text{ px}^2$

We evaluate bounding box area distributions across categories and inspect aspect ratios ($w/h$).


In [ ]:
def classify_coco_size(area: float) -> str:
    """Classifies bounding box area into standard COCO scale categories."""
    if area < 32 ** 2:
        return 'Small'
    elif area < 96 ** 2:
        return 'Medium'
    else:
        return 'Large'

df_all['coco_size'] = df_all['bbox_area'].apply(classify_coco_size)

# Tabulate COCO size breakdown per category
coco_size_table = pd.crosstab(df_all['category_name'], df_all['coco_size'], normalize='index') * 100
for col in ['Small', 'Medium', 'Large']:
    if col not in coco_size_table.columns:
        coco_size_table[col] = 0.0
coco_size_table = coco_size_table[['Small', 'Medium', 'Large']].reindex(CLASS_NAMES).fillna(0.0)

print('=== COCO Size Category Distribution (%) per Pathology ===')
print(coco_size_table.round(1))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Stacked Bar Chart of COCO scale categories
coco_palette = {'Small': '#4575b4', 'Medium': '#fdae61', 'Large': '#d73027'}
bottom = np.zeros(len(CLASS_NAMES))
for size_cat in ['Small', 'Medium', 'Large']:
    values = coco_size_table[size_cat].values
    axes[0].bar(CLASS_NAMES, values, bottom=bottom, label=size_cat, color=coco_palette[size_cat], edgecolor='black', alpha=0.85)
    bottom += values

axes[0].set_ylabel('Percentage of Annotations (%)')
axes[0].set_title('Pathology Bounding Box Scale Breakdown (COCO Metric)', fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].set_xticks(range(len(CLASS_NAMES)))
axes[0].set_xticklabels(CLASS_NAMES, rotation=15, ha='right')
axes[0].legend(title='COCO Scale', frameon=True)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# Plot 2: Bounding Box Dimensions (Width vs Height) Scatter & Aspect Ratio
for c in CLASS_NAMES:
    sub = df_all[df_all['category_name'] == c]
    axes[1].scatter(sub['bbox_w'], sub['bbox_h'], label=c, color=COLOR_MAP[c], alpha=0.7, edgecolors='white', s=45)

# Diagonal reference lines (aspect ratios 1:2, 1:1, 2:1)
max_dim = max(float(df_all['bbox_w'].max()) if len(df_all) else 100.0, float(df_all['bbox_h'].max()) if len(df_all) else 100.0, 100.0)
axes[1].plot([0, max_dim], [0, max_dim], 'k--', alpha=0.4, label='1:1 Aspect Ratio')
axes[1].plot([0, max_dim], [0, max_dim * 0.5], 'k:', alpha=0.4, label='2:1 Aspect Ratio')
axes[1].plot([0, max_dim * 0.5], [0, max_dim], 'k-.', alpha=0.4, label='1:2 Aspect Ratio')

axes[1].set_xlabel('Bounding Box Width (pixels)')
axes[1].set_ylabel('Bounding Box Height (pixels)')
axes[1].set_title('Bounding Box Geometry & Aspect Ratio Distribution', fontweight='bold')
axes[1].legend(loc='upper left', frameon=True)
axes[1].set_xlim(left=0)
axes[1].set_ylim(bottom=0)
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


## 4. 2D Spatial Pathology Heatmap on Panoramic Dental Arch Coordinates

Panoramic dental radiography (OPG - Orthopantomogram) maps a curved 3D anatomical structure onto a flat 2D plane:
- **Maxilla (Upper Jaw)**: Normalized vertical coordinates $y_c \in [0.20, 0.50]$
- **Mandible (Lower Jaw)**: Normalized vertical coordinates $y_c \in [0.50, 0.85]$
- **Posterior Molars / Angles**: Horizontal coordinates $x_c < 0.35$ (Right Quadrant) and $x_c > 0.65$ (Left Quadrant)
- **Anterior Incisors / Canines**: Midline horizontal coordinates $x_c \in [0.40, 0.60]$

Below, we plot normalized anatomical coordinates $(x_c, y_c) \in [0, 1] \times [0, 1]$ for each pathology class to reveal clinical spatial priors.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.flatten()

# Synthetic panoramic arch reference curve to guide anatomical interpretation
arch_x = np.linspace(0.1, 0.9, 100)
# Upper arch (maxilla)
arch_y_upper = 0.45 - 0.12 * np.sin(np.pi * (arch_x - 0.1) / 0.8)
# Lower arch (mandible)
arch_y_lower = 0.75 - 0.15 * np.sin(np.pi * (arch_x - 0.1) / 0.8)

for idx, cat_name in enumerate(CLASS_NAMES):
    ax = axes[idx]
    sub = df_all[df_all['category_name'] == cat_name]
    cat_color = COLOR_MAP[cat_name]
    
    # Anatomical reference curves
    ax.plot(arch_x, arch_y_upper, color='#888888', linestyle='--', linewidth=1.2, alpha=0.5, label='Maxillary Arch')
    ax.plot(arch_x, arch_y_lower, color='#555555', linestyle=':', linewidth=1.2, alpha=0.5, label='Mandibular Arch')
    ax.axvline(0.5, color='gray', linestyle='-.', alpha=0.3, label='Midline')
    
    # Density KDE overlay if sufficient points with variation exist
    if len(sub) >= 4 and sub['x_center_norm'].nunique() > 1 and sub['y_center_norm'].nunique() > 1:
        try:
            sns.kdeplot(
                data=sub,
                x='x_center_norm',
                y='y_center_norm',
                ax=ax,
                fill=True,
                cmap=sns.light_palette(cat_color, as_cmap=True),
                alpha=0.55,
                levels=5,
                thresh=0.08
            )
        except Exception:
            pass
            
    # Scatter points of lesion centers
    ax.scatter(
        sub['x_center_norm'],
        sub['y_center_norm'],
        c=cat_color,
        s=55,
        edgecolors='black',
        linewidth=0.8,
        alpha=0.85,
        label=f'{cat_name} (N={len(sub)})'
    )
    
    ax.set_title(f'2D Spatial Heatmap: {cat_name}', fontweight='bold', fontsize=12, color=cat_color)
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)  # Inverted so top is top of panoramic scan
    ax.set_xlabel('Normalized Horizontal Position ($x_c$)')
    ax.set_ylabel('Normalized Vertical Position ($y_c$)')
    ax.legend(loc='lower right', frameon=True, fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Panoramic Dental Arch 2D Spatial Pathology Heatmap\n(Revealing Anatomical Clustering and Clinical Priors)', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## 5. Pathology Co-occurrence & Correlation Matrix

Dental diseases frequently demonstrate clinical co-morbidity. For example:
- **Caries** progressing to **Deep Caries** and subsequently manifesting as **Periapical Lesion** via pulp necrosis.
- **Impacted Tooth** (such as third molars) creating food impaction niches and fostering caries in adjacent second molars.

We construct the binary scan-pathology co-occurrence matrix and compute cross-pathology Pearson correlation coefficients.


In [ ]:
# Construct scan-level multi-hot matrix
scans = df_all['image_id'].unique()
scan_cat_crosstab = pd.crosstab(df_all['image_id'], df_all['category_name']).clip(upper=1)
for c in CLASS_NAMES:
    if c not in scan_cat_crosstab.columns:
        scan_cat_crosstab[c] = 0
scan_cat_crosstab = scan_cat_crosstab[CLASS_NAMES]

# 1. Co-occurrence counts
co_occurrence_matrix = pd.DataFrame(0, index=CLASS_NAMES, columns=CLASS_NAMES)
for c1 in CLASS_NAMES:
    for c2 in CLASS_NAMES:
        co_occurrence_matrix.loc[c1, c2] = int(((scan_cat_crosstab[c1] == 1) & (scan_cat_crosstab[c2] == 1)).sum())

# 2. Conditional Probability P(Col | Row): Given Row is present, what is the probability Col also occurs?
row_totals = np.diag(co_occurrence_matrix.values)
cond_prob_matrix = co_occurrence_matrix.div(np.maximum(row_totals, 1), axis=0) * 100.0

# 3. Correlation Matrix
corr_matrix = scan_cat_crosstab.corr().fillna(0.0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap 1: Co-occurrence frequency with count annotations
sns.heatmap(
    co_occurrence_matrix,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    ax=axes[0],
    cbar=True,
    linewidths=1,
    linecolor='white'
)
axes[0].set_title('Pathology Co-occurrence Count Matrix\n(Scans Containing Both Pathologies)', fontweight='bold')
axes[0].set_xlabel('Co-occurring Pathology')
axes[0].set_ylabel('Primary Pathology')

# Heatmap 2: Pearson Correlation Heatmap
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    ax=axes[1],
    cbar=True,
    linewidths=1,
    linecolor='white'
)
axes[1].set_title('Cross-Pathology Pearson Correlation Matrix\n(Positive vs Negative Associations)', fontweight='bold')
axes[1].set_xlabel('Pathology')
axes[1].set_ylabel('Pathology')

plt.tight_layout()
plt.show()

print('=== Conditional Probability Matrix P(Col | Row) (%) ===')
print(cond_prob_matrix.round(1))


## 6. Radiographic Image Properties & Pixel Intensity Distribution

Panoramic dental radiographs feature distinct radiopacity patterns:
- **Enamel & Metallic Restorations**: Highly radiopaque (bright, high intensity ~180–255)
- **Dentin & Cortical Bone**: Moderately radiopaque (~100–180)
- **Dental Pulp & Soft Tissues**: Moderately radiolucent (~40–100)
- **Pathological Radiolucencies & Air Cavities (Maxillary Sinus, Pharynx)**: Strongly radiolucent (dark, low intensity ~0–50)

We profile image dimensions, aspect ratios, and intensity distributions across sample scans.


In [ ]:
image_paths = sorted(list(RAW_IMG_DIR.glob('*.png')) + list(RAW_IMG_DIR.glob('*.jpg')) + list(RAW_IMG_DIR.glob('*.jpeg')))
print(f'Total raw radiograph scans discovered: {len(image_paths)}')

img_stats = []
sample_intensities = []

# Analyze up to 20 representative scans
eval_paths = image_paths[:20] if len(image_paths) > 0 else []

for p in eval_paths:
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        h, w = img.shape
        img_stats.append({
            'file': p.name,
            'width': w,
            'height': h,
            'aspect_ratio': w / max(h, 1),
            'mean_intensity': float(img.mean()),
            'std_intensity': float(img.std()),
            'min_intensity': int(img.min()),
            'max_intensity': int(img.max()),
        })
        # Sample pixels for global histogram
        sampled = np.random.choice(img.flatten(), size=min(10000, img.size), replace=False)
        sample_intensities.extend(sampled)

img_stats_df = pd.DataFrame(img_stats)
if not img_stats_df.empty:
    print('\n=== Sample Radiograph Geometric & Contrast Summary ===')
    display(img_stats_df.head(5)) if 'display' in globals() else print(img_stats_df.head(5).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Histogram of pixel intensities
    axes[0].hist(sample_intensities, bins=64, range=(0, 255), color='#34495e', edgecolor='black', alpha=0.85, density=True)
    axes[0].axvline(50, color='#e74c3c', linestyle='--', label='Air / Radiolucent Boundary')
    axes[0].axvline(180, color='#27ae60', linestyle='--', label='Dentin / Enamel Boundary')
    axes[0].set_xlabel('8-bit Pixel Grayscale Intensity [0 - 255]')
    axes[0].set_ylabel('Probability Density')
    axes[0].set_title('Radiographic Pixel Intensity Distribution', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.5)

    # Boxplot of Image Contrast (Mean +/- Std)
    axes[1].errorbar(
        range(len(img_stats_df)),
        img_stats_df['mean_intensity'],
        yerr=img_stats_df['std_intensity'],
        fmt='o',
        color='#2980b9',
        ecolor='#7f8c8d',
        elinewidth=1.5,
        capsize=4
    )
    axes[1].set_xlabel('Scan Index')
    axes[1].set_ylabel('Mean Grayscale Intensity (± 1 Std)')
    axes[1].set_title('Radiographic Contrast & Dynamic Range Across Scans', fontweight='bold')
    axes[1].set_ylim(0, 255)
    axes[1].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()
else:
    print('No images found in raw directory to profile pixel intensity.')


## 7. Clinical Ground Truth Visualizer with Color-Coded Bounding Boxes

Finally, we provide an interactive visualizer to inspect sample panoramic radiographs with overlaid clinical ground truth bounding boxes color-coded by pathology category:
- **Caries**: `#E69F00`
- **Deep Caries**: `#D55E00`
- **Periapical Lesion**: `#0072B2`
- **Impacted Tooth**: `#009E73`


In [ ]:
def visualize_ground_truth_samples(
    coco_data: Dict[str, Any],
    raw_img_dir: Path,
    num_samples: int = 2,
    figsize: Tuple[int, int] = (16, 7)
):
    """Renders dental radiographs with ground truth color-coded pathology bounding boxes."""
    images = coco_data.get('images', [])
    annotations = coco_data.get('annotations', [])
    
    # Filter to images that have at least one annotation
    ann_img_ids = {a['image_id'] for a in annotations}
    valid_images = [im for im in images if im['id'] in ann_img_ids]
    
    if not valid_images:
        print('No annotated images available for visualization.')
        return
        
    display_images = valid_images[:num_samples]
    
    for im_info in display_images:
        img_path = raw_img_dir / im_info['file_name']
        if not img_path.exists():
            print(f'Image file not found on disk: {img_path}')
            continue
            
        bgr = cv2.imread(str(img_path))
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        
        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(rgb)
        
        scan_anns = [a for a in annotations if a['image_id'] == im_info['id']]
        
        for ann in scan_anns:
            cid = ann['category_id']
            c_meta = CLASS_METADATA.get(cid, {'name': f'Class {cid}', 'color': '#FFFFFF'})
            bx, by, bw, bh = ann['bbox']
            
            # Draw bounding box
            rect = patches.Rectangle(
                (bx, by), bw, bh,
                linewidth=2.5,
                edgecolor=c_meta['color'],
                facecolor='none'
            )
            ax.add_patch(rect)
            
            # Label banner
            ax.text(
                bx, max(by - 5, 15),
                c_meta['name'],
                color='white',
                fontsize=10,
                fontweight='bold',
                bbox=dict(facecolor=c_meta['color'], edgecolor='none', pad=2.5, alpha=0.9)
            )
            
        # Clinical legend
        legend_patches = [
            patches.Patch(color=CLASS_METADATA[k]['color'], label=CLASS_METADATA[k]['name'])
            for k in sorted(CLASS_METADATA.keys())
        ]
        ax.legend(handles=legend_patches, loc='upper right', framealpha=0.9, fontsize=10)
        
        ax.set_title(
            f"Panoramic Dental Radiograph: {im_info['file_name']} | Dimensions: {im_info['width']} x {im_info['height']} | Lesions: {len(scan_anns)}",
            fontweight='bold', fontsize=12
        )
        ax.axis('off')
        plt.tight_layout()
        plt.show()

# Visualize ground truth samples from train split
with open(TRAIN_JSON, 'r', encoding='utf-8') as f:
    train_coco_data = json.load(f)

visualize_ground_truth_samples(train_coco_data, RAW_IMG_DIR, num_samples=2)
